In [ ]:
import pandas as pd
from scipy.stats import linregress

def calcular_degradacao_pneus(csv_entrada, csv_saida):
    print("Calculando o desgaste por stint")

    # 1. Carrega a base estratégica
    df = pd.read_csv(csv_entrada, sep=';', decimal=',')

    # 2. Filtramos apenas as voltas válidas!
    # Ignoramos In-Laps, Out-Laps e Outliers (Safety Car, erros, etc)
    df_validas = df[(df['Tipo_Volta'] == 'Push') & (df['Outlier'] == False)].copy()

    # 3. Criamos um contador de "Volta dentro do Stint" (1, 2, 3...)
    df_validas['Volta_No_Stint'] = df_validas.groupby(['Carro', 'Stint']).cumcount() + 1

    resultados = []

    # 4. Agrupamos os dados por Carro e por Stint para analisar cada jogo de pneus
    for (carro, stint), grupo in df_validas.groupby(['Carro', 'Stint']):
        
        # No treino, aceitamos stints mais curtos (mínimo de 3 voltas limpas)
        if len(grupo) >= 3:
            x = grupo['Volta_No_Stint'].values
            y = grupo['Lap Tm (Segundos)'].values

            # Regressão Linear do SciPy
            slope, intercept, r_value, p_value, std_err = linregress(x, y)

            resultados.append({
                'Carro': carro,
                'Stint': stint,
                'Degradacao (s/volta)': round(slope, 3), 
                'Ritmo_Base (s)': round(intercept, 3),   
                'Voltas_Limpas': len(grupo),             
                'Consistencia_R2': round(r_value**2, 3)  
            })

    # 5. Transforma os resultados em uma tabela e salva
    df_deg = pd.DataFrame(resultados)
    df_deg.to_csv(csv_saida, index=False, sep=';', decimal=',')
    
    print(f"Cálculo finalizado! Relatório de degradação salvo em: {csv_saida}")

# --- ÁREA DE EXECUÇÃO ---
arquivo_estrategia = '../data/03_processed/TELEMETRIA_ESTRATEGIA_T2.csv'
arquivo_relatorio_deg = '../data/03_processed/RELATORIO_DEGRADACAO_T2.csv'

calcular_degradacao_pneus(arquivo_estrategia, arquivo_relatorio_deg)

Calculando o desgaste por stint
Cálculo finalizado! Relatório de degradação salvo em: ../data/03_processed/RELATORIO_DEGRADACAO_T2.csv


In [5]:
import pandas as pd
from scipy.stats import linregress

def calcular_degradacao_pura_treino(csv_entrada, constante_combustivel=0.015):
    print("Isolando Efeito Combustível nos TREINOS (P1)")
    print(f"Ganho estimado por peso: {constante_combustivel}s/volta")
    print("-" * 75)

    df = pd.read_csv(csv_entrada, sep=';', decimal=',')
    df_validas = df[(df['Tipo_Volta'] == 'Push') & (df['Outlier'] == False)].copy()
    df_validas['Volta_No_Stint'] = df_validas.groupby(['Carro', 'Stint']).cumcount() + 1

    resultados = []

    for (carro, stint), grupo in df_validas.groupby(['Carro', 'Stint']):
        # No treino, aceitamos stints mais curtos (mínimo de 3 voltas limpas)
        if len(grupo) >= 3:
            x = grupo['Volta_No_Stint'].values
            y_cronometro = grupo['Lap Tm (Segundos)'].values
            
            # Isolando o Pneu: Somamos a penalidade do peso para simular carro pesado
            y_pneu_puro = y_cronometro + (x * constante_combustivel)

            # Regressão 1: A Degradação Aparente (Cronômetro)
            slope_aparente, _, _, _, _ = linregress(x, y_cronometro)
            
            # Regressão 2: A Degradação Pura da Borracha (Física)
            slope_puro, intercept_puro, r_value, _, _ = linregress(x, y_pneu_puro)

            resultados.append({
                'Carro': carro,
                'Stint': stint,
                'Deg_Aparente (s/v)': round(slope_aparente, 3), 
                'Deg_Pura_Pneu (s/v)': round(slope_puro, 3), 
                'Ritmo_Base_Corrigido (s)': round(intercept_puro, 3),   
                'Voltas_Limpas': len(grupo)
            })

    df_deg_pura = pd.DataFrame(resultados)
    
    print(df_deg_pura.to_string(index=False))
    print("-" * 75)
    return df_deg_pura

# --- ÁREA DE EXECUÇÃO ---
# Apontando para os dados processados do Treino Livre 1 (P1)
arquivo_treino = '../data/03_processed/TELEMETRIA_ESTRATEGIA_P1.csv'

df_treino_puro = calcular_degradacao_pura_treino(arquivo_treino, constante_combustivel=0.015)

Isolando Efeito Combustível nos TREINOS (P1)
Ganho estimado por peso: 0.015s/volta
---------------------------------------------------------------------------
 Carro  Stint  Deg_Aparente (s/v)  Deg_Pura_Pneu (s/v)  Ritmo_Base_Corrigido (s)  Voltas_Limpas
     0      1               0.016                0.031                    83.484              5
     0      2               0.036                0.051                    82.522             15
     1      1              -0.036               -0.021                    82.728              5
     1      2               0.037                0.052                    82.265             15
     4      1              -0.181               -0.166                    83.643              8
     4      2               0.031                0.046                    82.560             12
     6      1              -0.336               -0.321                    85.764              9
     6      2              -0.013                0.002                   